In [ ]:
import sys
!git clone https://github.com/Ignas12345/masters_project_helper_functions.git
sys.path.append('/content/masters_project_helper_functions')

fatal: destination path 'masters_project_helper_functions' already exists and is not an empty directory.


In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.feature_selection import RFECV, RFE

In [ ]:
import pandas as pd
import numpy as np

import masters_project_helper_functions.utils as utils
import masters_project_helper_functions.feature_selection as feature_selection
import masters_project_helper_functions.preprocessing_methods as pp
import masters_project_helper_functions.classification_pipeline as classification_pipeline
import masters_project_helper_functions.results_overview as results_overview
import masters_project_helper_functions.plotting as plotting

In [ ]:
url_TGCT_non_TGCT_mirna_rpm_counts = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/expression_values/TGCT_non_TGCT_GTEx_combined_mirna_rpm_counts.csv"
url_TCGA_TGCT_divisions_by_experiment = "https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/sample_annotations/TCGA_TGCT_divisions_by_experiment.csv"
TGCT_non_TGCT_mirna_rpm_counts = pd.read_csv(url_TGCT_non_TGCT_mirna_rpm_counts, sep =';', index_col = 0)

experiment_label_dict = pd.read_csv(url_TCGA_TGCT_divisions_by_experiment, index_col=0)
TCGA_TGCT_samples = experiment_label_dict.index
gtex_samples = [sample for sample in TGCT_non_TGCT_mirna_rpm_counts.index if sample.startswith('GTEX')]
teratoma_burden_df = pd.read_csv('https://raw.githubusercontent.com/Ignas12345/masters_project_data_and_notebooks/refs/heads/main/Data/sample_annotations/teratoma_burden_df.csv', index_col =0)

In [ ]:
updated_teratoma_burden_df = teratoma_burden_df.copy()
for sample in gtex_samples:
  if sample not in teratoma_burden_df.index:
    updated_teratoma_burden_df.loc[sample] = [0,0,0]
updated_teratoma_burden_df.to_csv('updated_teratoma_burden_df.csv')

X = TGCT_non_TGCT_mirna_rpm_counts.loc[updated_teratoma_burden_df.index].copy()
y = updated_teratoma_burden_df['total_teratoma_burden'].values

pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          #'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,
                          }
feature_selection_method = feature_selection.rfecv_feature_selection
#housekeeping_gene = 'hsa-mir-16-1, mature,MIMAT0000069'
housekeeping_gene = 'hsa-mir-191, mature,MIMAT0000440'
feature_ranking_method = pp.perform_RFE_ranking
kwargs = {'housekeeping_list' : [housekeeping_gene, ],
          'scale_housekeep_by_mean' : False,
          #feature_ranking_method' : feature_ranking_method,
          #'keep_n_ranked_features' : 10,
          #'drop_correlated_features': True
          }
sample_label_dict = experiment_label_dict['teratoma_vs_non_teratoma'].to_dict()
for sample in X.index:
  if sample not in sample_label_dict.keys():
    sample_label_dict[sample] = 'non_teratoma'
kwargs = classification_pipeline.get_kwargs(sample_label_dict, kwargs)
X_pre_processed = classification_pipeline.run_pre_processing_on_train_set(X, pre_processing_methods, **kwargs)[0]

Running pre processing on train set
Running initial_feature_filtering: feature_filtering_by_class_means
number of features expressed (in mean) above 45 in class 1: 244
number of features expressed (in mean) above 45 in class 2: 284
number of features in kept across both classes: 331
Running sample_wise_scaling: normalize_by_housekeeping_list
Running feature_wise_scaling: log_normalization


In [ ]:
for i in np.arange(0,100, step=5):
  samples_with_teratoma_burden = teratoma_burden_df.index[teratoma_burden_df['total_teratoma_burden'] >= i]
  print(f'{len(samples_with_teratoma_burden)}' + f' with {i}')

137 with 0
30 with 5
29 with 10
28 with 15
27 with 20
26 with 25
24 with 30
22 with 35
22 with 40
21 with 45
21 with 50
19 with 55
18 with 60
17 with 65
17 with 70
16 with 75
16 with 80
11 with 85
10 with 90
8 with 95


In [ ]:
updated_teratoma_burden_df = teratoma_burden_df.copy()
for sample in gtex_samples:
  if sample not in teratoma_burden_df.index:
    updated_teratoma_burden_df.loc[sample] = [0,0,0]
updated_teratoma_burden_df.to_csv('updated_teratoma_burden_df.csv')
X = TGCT_non_TGCT_mirna_rpm_counts.loc[updated_teratoma_burden_df.index].copy()

pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          #'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,
                          }
y = updated_teratoma_burden_df['total_teratoma_burden'].values

feature_selection_method = feature_selection.rfecv_feature_selection
#housekeeping_gene = 'hsa-mir-16-1, mature,MIMAT0000069'
housekeeping_gene = 'hsa-mir-191, mature,MIMAT0000440'
feature_ranking_method = pp.perform_RFE_ranking
kwargs = {'housekeeping_list' : [housekeeping_gene, ],
          'scale_housekeep_by_mean' : False,
          #feature_ranking_method' : feature_ranking_method,
          #'keep_n_ranked_features' : 10,
          #'drop_correlated_features': True
          }

sample_label_dict = experiment_label_dict['teratoma_vs_non_teratoma'].to_dict()
for sample in X.index:
  if sample not in sample_label_dict.keys():
    sample_label_dict[sample] = 'non_teratoma'
kwargs = classification_pipeline.get_kwargs(sample_label_dict, kwargs)

X_pre_processed = classification_pipeline.run_pre_processing_on_train_set(X, pre_processing_methods, **kwargs)[0]


y = updated_teratoma_burden_df['total_teratoma_burden'].values
lasso = Lasso(alpha = 1)
RFE_selector = RFE(estimator=lasso, step=1, n_features_to_select = 1)
RFE_selector.fit(X_pre_processed, y)
ranking = pd.Series(RFE_selector.ranking_, index=X_pre_processed.columns)
ranking.sort_values(ascending=True, inplace=True)


Running pre processing on train set
Running initial_feature_filtering: feature_filtering_by_class_means
number of features expressed (in mean) above 45 in class 1: 244
number of features expressed (in mean) above 45 in class 2: 284
number of features in kept across both classes: 331
Running sample_wise_scaling: normalize_by_housekeeping_list
Running feature_wise_scaling: log_normalization


In [ ]:
ranking_list_1 = ranking.index.to_list()[:]

In [ ]:
#now perform RFECV on the top 10 features:
rfecv = RFECV(estimator=lasso, step=1, cv=5, scoring='neg_mean_squared_error')
rfecv.fit(X_pre_processed[ranking_list_1], y)

RFECV(cv=5, estimator=Lasso(alpha=1), scoring='neg_mean_squared_error')

In [ ]:
#get used features and their weights:
features = X_pre_processed[ranking_list_1].columns[rfecv.support_]
feature_weights_1 = rfecv.estimator_.coef_
feature_weights_1 = pd.Series(feature_weights_1, index=features)
feature_weights_1.rename('regression coefficients, TGCT + GTEx', inplace=True)
#feature_weights_1.to_latex('feature_weights_1.tex', float_format="%.2f")

,"regression coefficients, TGCT + GTEx"
"hsa-mir-199b, mature,MIMAT0000263",1.087862
"hsa-mir-99b, mature,MIMAT0000689",4.627212
"hsa-mir-372, mature,MIMAT0000724",-4.305360
"hsa-mir-199a-1, mature,MIMAT0000231",8.637201
"hsa-mir-215, mature,MIMAT0000272",2.008646
"hsa-mir-203a, mature,MIMAT0000264",2.678822
"hsa-mir-302b, mature,MIMAT0000715",-4.302321
"hsa-mir-514a-1, mature,MIMAT0002883",-1.544702
"hsa-mir-200c, mature,MIMAT0000617",1.445893
"hsa-mir-508, mature,MIMAT0002880",-0.623321


A version where we also include the GTEx samples

In [ ]:
updated_teratoma_burden_df = teratoma_burden_df.copy()
for sample in TGCT_non_TGCT_mirna_rpm_counts.index:
  if sample not in teratoma_burden_df.index:
    updated_teratoma_burden_df.loc[sample] = [0,0,0]
updated_teratoma_burden_df.to_csv('updated_teratoma_burden_df.csv')
X = TGCT_non_TGCT_mirna_rpm_counts.loc[updated_teratoma_burden_df.index].copy()
y = updated_teratoma_burden_df['total_teratoma_burden'].values

pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          #'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,
                          }

feature_selection_method = feature_selection.rfecv_feature_selection
#housekeeping_gene = 'hsa-mir-16-1, mature,MIMAT0000069'
housekeeping_gene = 'hsa-mir-191, mature,MIMAT0000440'
feature_ranking_method = pp.perform_RFE_ranking
kwargs = {'housekeeping_list' : [housekeeping_gene, ],
          'scale_housekeep_by_mean' : False,
          #feature_ranking_method' : feature_ranking_method,
          #'keep_n_ranked_features' : 10,
          #'drop_correlated_features': True
          }
sample_label_dict = experiment_label_dict['teratoma_vs_non_teratoma'].to_dict()
for sample in X.index:
  if sample not in sample_label_dict.keys():
    sample_label_dict[sample] = 'non_teratoma'
kwargs = classification_pipeline.get_kwargs(sample_label_dict, kwargs)
X_pre_processed = classification_pipeline.run_pre_processing_on_train_set(X, pre_processing_methods, **kwargs)[0]
print(X_pre_processed.shape)

lasso = Lasso()
RFE_selector = RFE(estimator=lasso, step=1, n_features_to_select = 1)
RFE_selector.fit(X_pre_processed, y)
ranking = pd.Series(RFE_selector.ranking_, index=X_pre_processed.columns)
ranking.sort_values(ascending=True, inplace=True)

Running pre processing on train set
Running initial_feature_filtering: feature_filtering_by_class_means
number of features expressed (in mean) above 45 in class 1: 244
number of features expressed (in mean) above 45 in class 2: 270
number of features in kept across both classes: 303
Running sample_wise_scaling: normalize_by_housekeeping_list
Running feature_wise_scaling: log_normalization
(508, 303)


In [ ]:
ranking_list_2 = ranking.index.to_list()[:]

In [ ]:
rfecv = RFECV(estimator=lasso, step=1, cv=5, scoring='neg_mean_squared_error')
rfecv.fit(X_pre_processed[ranking_list_2], y)
#get used features and their weights:
features = X_pre_processed[ranking_list_2].columns[rfecv.support_]
feature_weights_2 = rfecv.estimator_.coef_
feature_weights_2 = pd.Series(feature_weights_2, index=features)
feature_weights_2.rename('regression coefficients, TGCT + GTEx + other TCGA studies', inplace=True)
feature_weights_2.to_latex('feature_weights_2.tex', float_format="%.2f")
feature_weights_2

,"regression coefficients, TGCT + GTEx + other TCGA studies"
"hsa-mir-199b, mature,MIMAT0000263",14.240826
"hsa-mir-199a-1, mature,MIMAT0000231",3.873558
"hsa-mir-378a, mature,MIMAT0000732",-2.174801
"hsa-mir-215, mature,MIMAT0000272",1.363871


In [ ]:
feature_weights_1.to_latex('feature_weights_1_with_no_other_TCGA_samples.tex', float_format="%.2f")
feature_weights_2.to_latex('feature_weights_2_with_other_TCGA_samples.tex', float_format="%.2f")

Final experiment, where we only have teratoma samples and normal controls:

In [ ]:
updated_teratoma_burden_df = teratoma_burden_df.copy()
for sample in TGCT_non_TGCT_mirna_rpm_counts.index:
  if sample not in teratoma_burden_df.index:
    updated_teratoma_burden_df.loc[sample] = [0,0,0]
#updated_teratoma_burden_df.to_csv('updated_teratoma_burden_df.csv')
samples_to_include = [sample for sample in updated_teratoma_burden_df.index if updated_teratoma_burden_df.loc[sample, 'total_teratoma_burden'] > 0 or sample.startswith('GTEX')]

X = TGCT_non_TGCT_mirna_rpm_counts.loc[samples_to_include].copy()
y = updated_teratoma_burden_df.loc[samples_to_include, 'total_teratoma_burden'].values

pre_processing_methods = {'initial_feature_filtering' : pp.feature_filtering_by_class_means,
                          'sample_wise_scaling' : pp.normalize_by_housekeeping_list,
                          'feature_wise_scaling' : pp.log_normalization,
                          #'rank_features_and_keep_top_n' : pp.rank_features_and_keep_top_n_features,
                          }

feature_selection_method = feature_selection.rfecv_feature_selection
#housekeeping_gene = 'hsa-mir-16-1, mature,MIMAT0000069'
housekeeping_gene = 'hsa-mir-191, mature,MIMAT0000440'
feature_ranking_method = pp.perform_RFE_ranking
kwargs = {'housekeeping_list' : [housekeeping_gene, ],
          'scale_housekeep_by_mean' : False,
          #feature_ranking_method' : feature_ranking_method,
          #'keep_n_ranked_features' : 10,
          #'drop_correlated_features': True
          }
sample_label_dict = experiment_label_dict['teratoma_vs_non_teratoma'].to_dict()
new_sample_label_dict = {}
for sample in X.index:
  if sample not in sample_label_dict.keys():
    new_sample_label_dict[sample] = 'non_teratoma'
  else:
    new_sample_label_dict[sample] = sample_label_dict[sample]
sample_label_dict = new_sample_label_dict
kwargs = classification_pipeline.get_kwargs(sample_label_dict, kwargs)
X_pre_processed = classification_pipeline.run_pre_processing_on_train_set(X, pre_processing_methods, **kwargs)[0]
print(X_pre_processed.shape)


lasso = Lasso()
RFE_selector = RFE(estimator=lasso, step=1, n_features_to_select = 1)
RFE_selector.fit(X_pre_processed, y)
ranking = pd.Series(RFE_selector.ranking_, index=X_pre_processed.columns)
ranking.sort_values(ascending=True, inplace=True)

Running pre processing on train set
Running initial_feature_filtering: feature_filtering_by_class_means
number of features expressed (in mean) above 45 in class 1: 244
number of features expressed (in mean) above 45 in class 2: 221
number of features in kept across both classes: 317
Running sample_wise_scaling: normalize_by_housekeeping_list
Running feature_wise_scaling: log_normalization
(141, 317)


In [ ]:
ranking_list_3 = ranking.index.to_list()[:]
#ranking_list_3[:]

In [ ]:
rfecv = RFECV(estimator=lasso, step=1, cv=5, scoring='neg_mean_squared_error')
rfecv.fit(X_pre_processed[ranking_list_3], y)
#get used features and their weights:
features = X_pre_processed[ranking_list_3].columns[rfecv.support_]
feature_weights_3 = rfecv.estimator_.coef_
feature_weights_3 = pd.Series(feature_weights_3, index=features)
feature_weights_3.rename('regression coefficients, teratoma + GTEx', inplace=True)
feature_weights_3.to_latex('feature_weights_3.tex', float_format="%.2f")
feature_weights_3

,"regression coefficients, teratoma + GTEx"
"hsa-mir-199a-1, mature,MIMAT0000231",3.143121
"hsa-mir-302b, mature,MIMAT0000715",-9.071444
"hsa-mir-21, mature,MIMAT0000076",5.558634
"hsa-mir-372, mature,MIMAT0000724",-4.332266
"hsa-mir-99b, mature,MIMAT0000689",4.010952
"hsa-mir-127, mature,MIMAT0004604",3.557529
"hsa-mir-22, mature,MIMAT0000077",2.332009


In [ ]:
combined_weights = pd.concat([feature_weights_1, feature_weights_2, feature_weights_3], axis=1)
combined_weights.columns = ['TGCT + GTEx', 'TGCT + GTEx + TCGA other', 'teratoma + GTEx']
combined_weights.to_latex('combined_weights.tex', float_format="%.2f")
combined_weights

,TGCT + GTEx,TGCT + GTEx + TCGA other,teratoma + GTEx
"hsa-mir-199b, mature,MIMAT0000263",1.087862,14.240826,NaN
"hsa-mir-99b, mature,MIMAT0000689",4.627212,NaN,4.010952
"hsa-mir-372, mature,MIMAT0000724",-4.305360,NaN,-4.332266
"hsa-mir-199a-1, mature,MIMAT0000231",8.637201,3.873558,3.143121
"hsa-mir-215, mature,MIMAT0000272",2.008646,1.363871,NaN
"hsa-mir-203a, mature,MIMAT0000264",2.678822,NaN,NaN
"hsa-mir-302b, mature,MIMAT0000715",-4.302321,NaN,-9.071444
"hsa-mir-514a-1, mature,MIMAT0002883",-1.544702,NaN,NaN
"hsa-mir-200c, mature,MIMAT0000617",1.445893,NaN,NaN
"hsa-mir-508, mature,MIMAT0002880",-0.623321,NaN,NaN
